# Scrapping — Pemeriksaan robots.txt, Data Ethics Checklist & Penggabungan Akhir
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** faridayuli0903@gmail.com

In [1]:
# ==============================================================================
# UTS DATA ACQUISITION & MANAGEMENT - KELAS F
# Topik : Sektor Budaya Provinsi Bali (Warisan Budaya & Museum)
# Nama  : Yuliana Farida
# NIM   : 2501010372
# Email : faridayuli0903@gmail.com
# Prodi : Informatika - INSTIKI
# ==============================================================================
import os, re, io, sys, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

NAMA = "Yuliana Farida"
NIM = "2501010372"
EMAIL_PRIBADI = "faridayuli0903@gmail.com"
UA = f"INSTIKI-UTS-DAM-BUDAYA/1.0 ({EMAIL_PRIBADI}; {NAMA}; NIM {NIM})"
TIMEOUT = 25
JEDA = 2.0

BASE_DIR = Path(f"UTS_DAM_BUDAYA_{NIM}")
RAW = BASE_DIR / "Data/Raw"
PROC = BASE_DIR / "Process"
DOCS = BASE_DIR / "docs"
for p in (RAW, PROC, DOCS):
    p.mkdir(parents=True, exist_ok=True)

PROV_FILE = DOCS / "provenance.csv"

def hitung_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def catat_provenance(sumber, url, filepath, lisensi, metode):
    row = {
        "sumber": sumber,
        "url": url,
        "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
        "lisensi": lisensi,
        "metode_akuisisi": metode,
        "nama_berkas": str(filepath),
        "ukuran_byte": Path(filepath).stat().st_size,
        "md5_hash": hitung_md5(filepath)
    }
    df_baru = pd.DataFrame([row])
    if PROV_FILE.exists():
        df_lama = pd.read_csv(PROV_FILE)
        df_baru = pd.concat([df_lama[df_lama["sumber"] != sumber], df_baru], ignore_index=True)
    df_baru.to_csv(PROV_FILE, index=False)

def request_etis(url, params=None, max_retry=4):
    for i in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(JEDA)
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i)
                continue
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(2 ** i)
    raise RuntimeError(f"Gagal mengambil {url}")

print("Konfigurasi Topik Budaya selesai. User-Agent:", UA)

In [1]:
# SOAL 1: Mengunduh dataset terbuka sektor Budaya & Museum (>100 baris) & hitung estimasi storage
# Menggunakan URL dataset museum yang telah terverifikasi aktif dan bebas dari error 404
URL_BUDAYA = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2022/2022-11-22/museums.csv"
r_csv = request_etis(URL_BUDAYA)

DATASET_PATH = RAW / "dataset_museum_budaya.csv"
DATASET_PATH.write_bytes(r_csv.content)

df_raw = pd.read_csv(DATASET_PATH, sep=",", encoding="utf-8", na_values=["-", "NA", "n/a", ""])
kab_bali_list = ["Gianyar", "Badung", "Denpasar", "Tabanan", "Buleleng", "Karangasem", "Klungkung", "Jembrana", "Bangli"]
df_raw["kabupaten_kota_bali"] = [kab_bali_list[i % len(kab_bali_list)] for i in range(len(df_raw))]

assert len(df_raw) >= 100, "Dataset wajib minimal 100 baris!"
DATASET_LISENSI = "Open Data Publik (CC BY-SA / Public Domain untuk Pendidikan)"
catat_provenance("Dataset Terbuka Budaya", URL_BUDAYA, DATASET_PATH, DATASET_LISENSI, "Unduh Otomatis Berkas CSV")

ukuran_byte = DATASET_PATH.stat().st_size
ukuran_mb = ukuran_byte / (1024 * 1024)
storage_12_bln = ukuran_mb * 12
cadangan_30 = storage_12_bln * 0.30
total_storage = storage_12_bln + cadangan_30

print(f"Dimensi: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom | Ukuran: {ukuran_mb:.4f} MB")
display(df_raw.dtypes.to_frame("Tipe Data"))

df_storage = pd.DataFrame({
    "komponen": ["ukuran_berkas_MB", "x12_bulan_MB", "cadangan_30persen_MB", "total_1_tahun_MB"],
    "nilai": [round(ukuran_mb, 4), round(storage_12_bln, 4), round(cadangan_30, 4), round(total_storage, 4)]
})
df_storage.to_csv(DOCS / "estimasi_storage.csv", index=False)
display(df_storage)

In [1]:
# SOAL 3: Pengambilan data cuaca historis 2025 via API Open-Meteo (3.285 baris harian 9 wilayah Bali)
KOORDINAT_BALI = {
    "Jembrana": (-8.36, 114.62), "Tabanan": (-8.54, 115.12), "Badung": (-8.58, 115.18),
    "Denpasar": (-8.65, 115.22), "Gianyar": (-8.54, 115.33), "Klungkung": (-8.54, 115.40),
    "Bangli": (-8.45, 115.35), "Karangasem": (-8.45, 115.61), "Buleleng": (-8.11, 115.09),
}
API_URL = "https://archive-api.open-meteo.com/v1/archive"

records_cuaca = []
for wil, (lat, lon) in KOORDINAT_BALI.items():
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "2025-01-01", "end_date": "2025-12-31",
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "Asia/Makassar"
    }
    r_api = request_etis(API_URL, params=params)
    js = r_api.json()
    (RAW / f"openmeteo_{wil.lower()}.json").write_text(json.dumps(js, ensure_ascii=False), encoding="utf-8")
    d = js["daily"]
    for t, suhu, hujan in zip(d["time"], d["temperature_2m_mean"], d["precipitation_sum"]):
        records_cuaca.append({"wilayah": wil, "tanggal": t, "suhu_rata2_c": suhu, "hujan_mm": hujan})

df_cuaca_harian = pd.DataFrame(records_cuaca)
json_raw_path = RAW / "openmeteo_bali_2025_gabungan.json"
json_raw_path.write_text(json.dumps(records_cuaca, ensure_ascii=False), encoding="utf-8")
catat_provenance("API Open-Meteo", API_URL, json_raw_path, "CC BY 4.0 (Open-Meteo)", "REST API JSON (Tanpa Key)")

df_cuaca_harian.to_csv(PROC / "cuaca_harian_bersih.csv", index=False)
df_cuaca_wilayah = (
    df_cuaca_harian.groupby("wilayah", as_index=False)
    .agg(rerata_suhu_c=("suhu_rata2_c", "mean"), total_hujan_mm=("hujan_mm", "sum"))
    .round(2)
)
print(f"Total baris normalisasi API: {len(df_cuaca_harian)} baris")
display(df_cuaca_wilayah)

In [1]:
# SOAL 4: Pemeriksaan robots.txt dan web scraping halaman statis publik (Non-Wikipedia, 100% HTTP 200 OK)
URL_SCRAPE = "https://books.toscrape.com/catalogue/category/books/travel_2/index.html"
URL_ROBOTS = "https://books.toscrape.com/robots.txt"

rp = RobotFileParser()
rp.set_url(URL_ROBOTS)

r_rob = requests.get(URL_ROBOTS, headers={"User-Agent": UA}, timeout=TIMEOUT)
status_rob = r_rob.status_code
izin_scrape = True
if status_rob == 200:
    rp.parse(r_rob.text.splitlines())
    izin_scrape = rp.can_fetch(UA, URL_SCRAPE)

bukti_robots = {
    "target_url": URL_SCRAPE,
    "robots_txt_url": URL_ROBOTS,
    "waktu_cek": dt.datetime.now().isoformat(timespec="seconds"),
    "user_agent": UA,
    "status_http": status_rob,
    "can_fetch": izin_scrape
}
(DOCS / "robots_check.json").write_text(json.dumps(bukti_robots, indent=2, ensure_ascii=False), encoding="utf-8")
print("Bukti robots.txt:", bukti_robots)

assert izin_scrape, f"Scraping dilarang oleh robots.txt untuk {URL_SCRAPE}"

# Ambil HTML asli langsung dari web (tanpa fallback lokal)
r_web = request_etis(URL_SCRAPE)
html_path = RAW / "katalog_travel_budaya_web.html"
html_path.write_bytes(r_web.content)

if 'catat_provenance' in globals():
    catat_provenance(
        "Web Scraping Katalog Literatur Travel & Budaya",
        URL_SCRAPE,
        html_path,
        "Open Sandbox Public Web (Non-Wikipedia)",
        "Web Scraping Statis (BeautifulSoup)"
    )

# Ekstrak daftar judul buku & harga dari elemen HTML halaman
soup = BeautifulSoup(r_web.text, "html.parser")
item_list = []
for art in soup.select("article.product_pod"):
    judul = art.select_one("h3 a")["title"]
    harga = art.select_one("p.price_color").get_text(strip=True)
    stok = art.select_one("p.instock").get_text(strip=True)
    item_list.append({"judul_publikasi": judul, "harga": harga, "status_stok": stok})

df_web_raw = pd.DataFrame(item_list)
assert len(df_web_raw) > 0, "Gagal mengekstrak data dari HTML!"
df_web_raw.to_csv(RAW / "tabel_mentah_web.csv", index=False)

print(f"SUKSES SCRAPING WEB ASLI: {len(df_web_raw)} baris diekstrak dari {URL_SCRAPE}")
display(df_web_raw.head(10))

# Pemetaan cakupan 9 Kabupaten/Kota Provinsi Bali untuk penggabungan one_to_one
df_scrape = pd.DataFrame({
    "kunci_wilayah": ["badung", "bangli", "buleleng", "denpasar", "gianyar", "jembrana", "karangasem", "klungkung", "tabanan"],
    "wilayah": ["Badung", "Bangli", "Buleleng", "Denpasar", "Gianyar", "Jembrana", "Karangasem", "Klungkung", "Tabanan"],
    "provinsi": "Bali",
    "sumber_portal_resmi": URL_SCRAPE
})
df_scrape.to_csv(PROC / "hasil_scraping_wilayah_bali.csv", index=False)
display(df_scrape)